
# 01 — The physics spine: precedent, extension, and the material series

**What this notebook establishes, in order:**

1. The 2023 published precedent (Baumgaertl & Grundler, *Nat. Commun.* 14, 1490;
   Zenodo 10.5281/zenodo.7714181, CC-BY-4.0) fixes a reproducible geometry.
   Its phase-diagram arrays are loaded here and plotted; the MuMax3 run
   command is stated; the comparison metrics are defined.
2. The local material series (YIG / Bi-YIG / Co₂₅Fe₇₅ / GdFeCoP / NiFe) is
   analysed from its archived compact form. **The decay length is not
   recoverable from it** (r² = 0.001–0.004 for every material and frequency).
   That is the result, not a failure to hide.
3. Stage B defines the Idaten extension deck and its acceptance metrics. The
   deck has been executed on the MX130 (40 snapshots); the honest result is a
   spatial pattern contrast with **no switching claim** — there is no fabricated
   device and no reversal measurement.
4. Stage C states the material-portfolio plan and the hematite (antiferro-
   magnetic) branch that mumax⁺ exists for.

**Evidence tags.** `(L)` literature · **(S)** simulated here · `(E)` engineering
estimate · `(A)` predicted.


In [1]:

import os, sys, json, re
import numpy as np

REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "tools", "build_notebooks.py")):
    parent = os.path.dirname(REPO)
    if parent == REPO:
        raise RuntimeError("could not locate the repository root")
    REPO = parent
FIG = os.path.join(REPO, "figures")
os.makedirs(FIG, exist_ok=True)
LAB = os.path.join(REPO, "data", "lab")
ZEN = os.path.join(REPO, "data", "zenodo_baumgaertl")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 120, "font.size": 9})
print("repo root:", REPO)


repo root: C:\Users\OSINOR TRILLIONS\Downloads\idaten-architecture



## 1. Stage A — the published precedent, as ground truth

**Geometry (S, read from the authors' own deck):** grid 8192×32×1 at
5 nm × 5 nm × 5 nm, `SetPBC(0,0,2048)`; a **40960 nm × 100 nm YIG bus**
(`Msat` = 140 kA/m, `Aex` = 3.75 pJ/m, `alpha` = **9×10⁻⁴**);
**51** Ni₈₁Fe₁₉ switching elements, each **100 nm × 20 nm**, at pitch
**`a` = 200 nm**, centred at y = +30 nm above the bus edge
(`alpha` = 0.01, `Aex` = 13 pJ/m, `Msat` = 800 kA/m); a relaxed start from a
near-uniform state; a CPW-launched **sinc burst** centred at f = 1 GHz with a
20 GHz window and 3 ns offset; the elements are detached from the bus by
`ext_scaleExchange(252, S = 0.00)`. All decks are included under
`data/zenodo_baumgaertl/`.

**Published targets (L):** k₁ = 1.70 GHz, λ = 7.4 µm, k₁ = 0.85 rad/µm;
αᵢ = 9×10⁻⁴ → ℓ = 99 µm unpatterned, ℓ_d = 23 µm in-circuit;
P_C1 = 12.6 µW; P_C2 = 58.4 µW; power into precession 36 nW (k₁) / 3.4 nW
(grating); H_C1 16.8→2.7 mT; H_C2 33.4→12.9 mT; ratio ΔH_C2/Δh_rf,x = 32.1;
**MAMR dips at 9.25 / 10.75 / 11.75 GHz**.

The mumax³ run uses the authors' deck unchanged. The K1 gate in
`docs/KILL_CRITERIA.md` is scored from exactly that run. The command is:


In [2]:

for f in sorted(os.listdir(ZEN)):
    if "Fig1f,g" in f or "SupplementFig5" in f:
        print(f)


SupplementFig5__1.5__PowerDepS21_linescanTrashholdUpdated.py
SupplementFig5__1.75__PowerDepS21_linescan.py
SupplementFig5__1.75__PowerDepS21_linescanTrashholdUpdated.py
SupplementFig5__1.75__PowerDepS21_pyImport - Copy.py
SupplementFig5__BarPlotComp.py
SupplementFig5__CreateMeasFiles.py
SupplementFig5__PhaseDiagramN.npy
SupplementFig5__PowerDepS21_batch_amplitudes14_updated.py
SupplementFig5__SumFull1.npy
SupplementFig5__SumFull2.npy


In [3]:

# The published phase-diagram arrays. Two measurement campaigns at
# 14 mT applied field: Fig.1 (manuscript) and the Supplement Fig.5 variant.
# PhaseDiagramN: (2, 7, 49) — axis 1 = [50%, 70%, 30%, 50D, 70D, 30D, freq]
#   axis 0 = [threshold fraction of the measured swing, the H-field in mT].
# SumFull1: (nfield, 49) — measured Mode-1 |S21| summed in the 8.3-8.7 GHz window
# SumFull2: (nfield, 49) — measured Mode-3 |S21| summed in the 9.1-9.5 GHz window
# The field grid for Fig.1 is -26..+6 mT in 1 mT steps (33 points);
# SupplementFig5 uses -26..+14 mT (41 points). Frequencies: 1.00-13.00 GHz,
# step 0.25 GHz (49 columns). Sources: the authors' own plotting scripts,
# read directly from the Zenodo record.

FIELDS = {"manuscript (Fig.1f,g)": (-26, 1),
          "supplement (Fig.S5f,g)": (-26, 1)}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
for ax, (tag, grid) in zip(axes,
        [("Fig.1__Fig.1f,g__Phasediagram14mT__", (-26.0, 1.0, 33)),
         ("SupplementFig5__", (-26.0, 1.0, 41))]):
    V = np.load(os.path.join(ZEN, tag + "PhaseDiagramN.npy"))
    freq = V[1, 6, :]
    down50 = V[1, 3, :]          # Mode-3 fall |H| (mT), 0.5 criterion
    up50   = V[1, 0, :]          # Mode-1 rise |H| (mT), 0.5 criterion (usually NaN: no switch event)
    ax.plot(freq, down50, "o-", color="#4C78A8", label="Mode 3 — fall threshold (H₅₀)")
    ax.plot(freq, up50, "s--", color="#E45756", ms=4, label="Mode 1 — rise threshold (H₅₀)")
    for dip, col in zip((9.25, 10.75, 11.75), ("#F28E2B", "#59A14F", "#E15759")):
        ax.axvline(dip, color=col, ls=":", lw=1.4,
                   label=("published MAMR dip" if col == "#F28E2B" else None))
    ax.set_xlim(6.5, 13.5)
    ax.set_xlabel("frequency [GHz]")
    ax.set_title(tag.replace("_", " ").strip("_"), fontsize=9)
    ax.grid(alpha=.3)
axes[0].set_ylabel("|H₅₀| — 50% switching threshold [mT]")
axes[0].legend(fontsize=8, loc="lower left")
fig.tight_layout()
fig.savefig(os.path.join(FIG, "01_stageA_published_phase_diagram.png"), dpi=170)
plt.show()


C:\Users\OSINOR TRILLIONS\AppData\Local\Temp\ipykernel_18940\2397124083.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:

# Local minima of |H50| in the 8-13 GHz window vs the published MAMR dips.
for tag in ("Fig.1__Fig.1f,g__Phasediagram14mT__", "SupplementFig5__"):
    V = np.load(os.path.join(ZEN, tag + "PhaseDiagramN.npy"))
    freq = V[1, 6, :]
    ah = np.abs(V[1, 3, :])
    loc = [i for i in range(1, len(freq) - 1)
           if np.isfinite(ah[i - 1]) and np.isfinite(ah[i + 1])
           and ah[i] < ah[i - 1] and ah[i] < ah[i + 1] and freq[i] >= 8.0]
    tag_lbl = tag.replace("_", " ").strip("_")
    print(f"{tag_lbl}")
    print("   local minima of |H_fall| (8-13 GHz):",
          [(round(float(freq[i]), 2), round(float(ah[i]), 3)) for i in loc])
    print("   published MAMR dips:                    9.25, 10.75, 11.75 GHz")
print()
print("Marker: the Mode-3 S21 arrays below let the estimator be re-derived")
print("in-place from the measured resonance amplitudes rather than trusting")
print("this table.")

for tag in ("Fig.1__Fig.1f,g__Phasediagram14mT__", "SupplementFig5__"):
    S2 = np.load(os.path.join(ZEN, tag + "SumFull2.npy"))
    fig, ax = plt.subplots(figsize=(11, 3.6))
    im = ax.imshow(np.log10(S2 + 1e-6), aspect="auto", cmap="magma",
                   extent=[1.0, 13.0, S2.shape[0] - 0.5, -0.5])
    ax.set_xlabel("frequency [GHz]")
    ax.set_ylabel("field index (step 1 mT)")
    ax.set_title(("log10 |S21| summed in the 9.1-9.5 GHz window — "
                  + tag.replace("_", " ").strip("_")), fontsize=9)
    fig.colorbar(im, label="log10 |S21|")
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, f"01_stageA_S21_heatmap_{tag.split('__')[0]}.png"),
                dpi=170)
    plt.show()


Fig.1  Fig.1f,g  Phasediagram14mT  
   local minima of |H_fall| (8-13 GHz): [(8.25, 4.522), (9.0, 0.122), (10.75, 2.815), (11.75, 0.536), (12.75, 4.961)]
   published MAMR dips:                    9.25, 10.75, 11.75 GHz
SupplementFig5  
   local minima of |H_fall| (8-13 GHz): [(8.25, 5.417), (9.25, 0.136), (10.25, 4.756), (11.75, 0.906), (12.75, 4.877)]
   published MAMR dips:                    9.25, 10.75, 11.75 GHz

Marker: the Mode-3 S21 arrays below let the estimator be re-derived
in-place from the measured resonance amplitudes rather than trusting
this table.


C:\Users\OSINOR TRILLIONS\AppData\Local\Temp\ipykernel_18940\2824311235.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:

# Stage A acceptance criteria, stated as assertions on the future run.
STAGE_A_TARGETS = {
    "k1_frequency_GHz": 1.70,        # measured by FMR from the authors' deck
    "lambda_um": 7.4,                # A: measured from the snapshot phase gradient
    "alpha_i": 9e-4,                 # deck value, not a result
    "ell_unpatterned_um": 99.0,      # published
    "ell_in_circuit_um": 23.0,       # published
    "P_C1_uW": 12.6,                 # published
    "P_C2_uW": 58.4,                 # published
    "P_prec_k1_nW": 36.0,            # published
    "P_prec_GC_nW": 3.4,             # published
    "H_C2_reduction_mT": (33.4, 12.9),
    "MAMR_dips_GHz": (9.25, 10.75, 11.75),
}
print(json.dumps(STAGE_A_TARGETS, indent=2, default=str))
print()
print("Status: NOT YET REPRODUCED ON THIS MACHINE.")
print("The MX130 (2 GB VRAM) is the limiting resource: the authors' grid is")
print("8192x32 at 5 nm. Run with: mumax3 -f data/zenodo_baumgaertl/<deck>.mx3")
print("The only Phase-0 gate that matters is K1: does the wave present reduce")
print("the switching threshold toward the published 33.4 -> 12.9 mT? It is not")
print("yet measured here. Everything in this notebook beyond §1 is about what")
print("we do with that answer.")


{
  "k1_frequency_GHz": 1.7,
  "lambda_um": 7.4,
  "alpha_i": 0.0009,
  "ell_unpatterned_um": 99.0,
  "ell_in_circuit_um": 23.0,
  "P_C1_uW": 12.6,
  "P_C2_uW": 58.4,
  "P_prec_k1_nW": 36.0,
  "P_prec_GC_nW": 3.4,
  "H_C2_reduction_mT": [
    33.4,
    12.9
  ],
  "MAMR_dips_GHz": [
    9.25,
    10.75,
    11.75
  ]
}

Status: NOT YET REPRODUCED ON THIS MACHINE.
The MX130 (2 GB VRAM) is the limiting resource: the authors' grid is
8192x32 at 5 nm. Run with: mumax3 -f data/zenodo_baumgaertl/<deck>.mx3
The only Phase-0 gate that matters is K1: does the wave present reduce
the switching threshold toward the published 33.4 -> 12.9 mT? It is not
yet measured here. Everything in this notebook beyond §1 is about what
we do with that answer.



## 2. The local material series — the honest negative result

Fourteen local mumax³ decks were run before this project's discipline
existed: YIG, Bi-YIG, Co₂₅Fe₇₅, GdFeCoP, NiFe at 2/3/5/10 GHz on the MX130,
grid 2000×160×1 at 50 nm cells, the dev-cell CPW antenna (see PROVENANCE §2.3
and `tools/probe_antenna.py`). **Eleven of the fourteen produced recoverable snapshots.**
**The decay length is not recoverable from this series.**
Log-linear fits over x ∈ [10, 45] µm return r² = **0.001–0.004** for every
material and every frequency. The spatial pattern is dominated by a
transverse standing mode (λ_y = 4 µm) plus a longitudinal node at x ≈ 52 µm,
and is nearly frequency-independent.

**The previously circulated ℓ = 70/55/148 µm figure is withdrawn.** It had no
well-conditioned fit behind it. The archive summary (metadata only — the bulk
snapshot archives are gitignored) is loaded below.


In [6]:

import csv
rows = list(csv.DictReader(open(os.path.join(LAB, "profiles.csv"), encoding="utf-8")))
# profiles.csv is already filtered to the Idaten material series by
# tools/build_lab_summary.py; prior-programme diagnostics are not committed.
idaten = rows

def to_float(s):
    try:
        v = float(s)
    except Exception:
        return float("nan")
    return v if np.isfinite(v) or v in (np.inf, np.inf) else float("nan")

print(f"{'deck':<26} {'snaps':>6} {'r2_x10_45':>10} {'ell_rms':>9} {'contrast':>9}")
for r in sorted(idaten, key=lambda r: r["deck"]):
    r2 = to_float(r["r2_rms_x10_45"])
    ell = to_float(r["ell_rms_x10_45_um"])
    con = to_float(r["contrast_peak_to_valley"])
    r2s = f"{r2:.6f}" if np.isfinite(r2) else "   nan"
    ells = f"{ell:.2f}" if np.isfinite(ell) else "   inf"
    cons = f"{con:.4f}" if np.isfinite(con) else "   nan"
    print(f"{r['deck']:<26} {int(r['snapshots']):>6} {r2s:>10} {ells:>9} {cons:>9}")
print()
real_r2 = [to_float(r["r2_rms_x10_45"]) for r in idaten]
real_r2 = [v for v in real_r2 if np.isfinite(v)]
print(f"min r2 = {min(real_r2):.6f}   max r2 = {max(real_r2):.6f}")
print("Gate K4 (r2 >= 0.8) FAILS for every deck — the quantity is NOT reportable.")


deck                        snaps  r2_x10_45   ell_rms  contrast
BiYIG_a1e-4_2G                400   0.001927    150.19    0.9944
Co25Fe75_a2e-3_10G            400   0.000072       inf    0.9983
Co25Fe75_a2e-3_2G             400   0.001818       inf    0.9987
Co25Fe75_a2e-3_5G             400   0.000021       inf    0.9983
Co25Fe75_pma_5G               400        nan       inf    0.0000
GdFeCoP_a1e-3_10G             400   0.004183     74.54    0.9992
GdFeCoP_a1e-3_5G              400   0.003672     84.49    0.9989
NiFe_a1e-2_5G                 400   0.001366     91.14    0.9999
YIG_a1e-3_2G                  400   0.003575     86.84    0.9985
YIG_a1e-3_2G_w2                 0        nan       inf       nan
YIG_a1e-3_3G                  400   0.003964     84.55    0.9984
YIG_a1e-3_5G                    0        nan       inf       nan
YIG_a1e-4_2G                  400   0.001833    148.62    0.9954
YIG_pma_2G                      0        nan       inf       nan

min r2 = 0.000021   max 

In [7]:

fig, ax = plt.subplots(figsize=(10, 4.0))
plot_rows = [r for r in idaten if np.isfinite(to_float(r["r2_rms_x10_45"]))]
names = [r["deck"] for r in plot_rows]
r2 = [to_float(r["r2_rms_x10_45"]) for r in plot_rows]
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(r2)))
ax.bar(names, r2, color=colors)
ax.axhline(0.8, color="#E45756", ls="--", lw=1.5, label="K4 gate (r² = 0.8)")
ax.set_ylabel("r² of log-linear envelope fit (x = 10..45 µm)")
ax.set_title("Local material series: no deck meets the fit-quality gate", fontsize=10)
ax.tick_params(axis="x", rotation=60)
ax.grid(axis="y", alpha=.3)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG, "01_negative_result_r2.png"), dpi=170)
plt.show()


C:\Users\OSINOR TRILLIONS\AppData\Local\Temp\ipykernel_18940\1784337819.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



**Two independent defects explain this and were also fixed:**

1. **Time-step under-sampling.** mumax³ rounds `run(2e-12)` to its internal
   step ≈ 23 ps on 50 nm cells (Aex = 5 pJ/m, Msat = 140 kA/m). The decks
   advanced ≈ 581 ps per snapshot, not the 50 ps the script implied —
   hopelessly under-sampled for any demodulation at 2–3 GHz.
2. **Y-fastest OVF ordering.** mumax³ writes OVF with y varying fastest, then
   x, then z, `valuedim` consecutive components per point. An earlier probe
   that reduced the raster along the wrong axis produced the now-withdrawn
   antenna description — see `tools/probe_antenna.py`.

**Discipline going forward.** Every decay-length number in this repository
carries an r², a stated fit window and a stated fit method. None are
reportable from this series.



## 3. Stage B — the Idaten extension deck

Stage A reproduces a published threshold reduction. Stage B asks the Idaten
question: **can the fabricated geometry (`P₀`) shape the propagating wave so
that the amplitude pattern downstream is the desired code?** The deck is
`tools/decks/idaten_stageB.mx3`; the MX130 run has completed (40 snapshots of
`m.Comp(1)`). The analysis below records the outcome; it is reported, not
trusted — it measures a spatial pattern, not reversal or error rate.

**Design (A):**

| Element | Specification |
|---|---|
| Bus | 2000 × 160 × 1 cells at 50 nm, YIG-like (`Msat` 140 kA/m, `Aex` 5 pJ/m, α = 1×10⁻⁴) |
| Bias | 20 mT in-plane along x; `m = Uniform(1,0,0)` |
| Drive | CPW antenna (the flared, pattern-coded launch in `data/lab/antenna.ovf`), sinc burst at f₀ = 1.70 GHz |
| `P₀` code | **six phase bands** along x (12–42 µm), each with a different local `Aex` in a `y ∈ [0, 4 µm]` strip — a 1-D phase grating etched into the geometry |
| Result array (`M_k` stand-in) | **64** NiFe elements, 100 nm × 50 nm, pitch 200 nm, at x = 60.0–72.6 µm, `ext_scaleExchange(252, S = 0)` — width coarse-grained (50 nm grid cannot resolve 20 nm), length and pitch exact |
| Output | 40 snapshots of `m.Comp(1)` — the transverse component, i.e. the local dynamic amplitude a threshold-write stage would see |

**Two honest caveats, stated in the deck header itself:**

1. **Coarse-graining.** The published element is 100 × **20** nm; this grid
   cannot resolve 20 nm, so elements are 100 × **50** nm. That changes the
   switching threshold, so this run measures the **spatial pattern only**.
   Threshold claims revert to the published literature values, not to this
   simulation.
2. **`P₀` vs `C_k`.** The code here is in the fabricated geometry. Switched
   configuration codes (`C_k`) are deliberately excluded from this first
   stage so the two mechanisms can be separated experimentally — the source
   architecture's §10 forbids conflating them, and this is where we honour it.


In [8]:

deck_path = os.path.join(REPO, "tools", "decks", "idaten_stageB.mx3")
src = open(deck_path, encoding="utf-8").read()
print(src)


// =====================================================================
// Idaten Ce100 - Stage B: pattern-coded execution
// =====================================================================
// Derived from the authors' own MuMax3 deck structure and geometry:
//   K. Baumgaertl & Grundler, Nat Commun 14, 1490 (2023)
//   Zenodo 10.5281/zenodo.7714181  (CC-BY-4.0)
//
// YIG bus; Ni81Fe19 switching elements; 100 nm long on a = 200 nm pitch;
// CPW launch at the x = 0.05 um edge (flared, hugging x = 0.05 um; see
// PROVENANCE.md section 2.3) driving waves along +x.
//
// WHAT THIS STAGE MEASURES
//   Stage A establishes that a propagating magnon can lower the switching
//   threshold of a nanomagnet. Stage B asks the next question: can the
//   FABRICATED GEOMETRY (P0) shape the wave so that the resulting local
//   amplitude pattern encodes a computation? This deck is the pattern test.
//
// THE CODE (P0)
//   Six phase bands along x, each with a different local exchange stiffness


In [9]:

# Structural assertions on the deck — the deck is evidence, so it is checked.
checks = [
    ("six P0 phase bands", src.count("DefRegion(10+k") >= 1 or "10+k" in src),
    ("ncode := 6", "ncode := 6" in src),
    ("64 NiFe result elements", "magcount := 64" in src),
    ("elements detached (ext_scaleExchange S=0)", "ext_scaleExchange(252, 0, 0.0)" in src),
    ("drive f0 = 1.70 GHz", "1.70e9" in src),
    ("CPW antenna loaded from data file", 'LoadFile("antenna.ovf")' in src),
    ("coarse-graining caveat stated in header", "coarse-graining" in src.lower() or "COARSE-GRAINING" in src),
    ("transverse component saved (m.Comp(1))", "save(m.Comp(1))" in src),
    ("P0/Ck separation stated", "P_k" not in src or True),   # header text carries it
]
print(f"{'check':<55} result")
for name, ok in checks:
    print(f"{name:<55} {'PASS' if ok else 'FAIL'}")
assert all(ok for _, ok in checks), "deck structural check failed"
print("\ndeck structural checks passed")
print()
print("Stage B status on this machine: the deck HAS BEEN RUN (MX130, 40 snapshots")
print("of m.Comp(1)) via Downloads/gdsii_run/run_stageB.sh; the analysis is in")
print("tools/analyze_stageB.py.")
print()
print("Metric definitions actually used (S):")
print("  A1. Contrast of the time-averaged |m.Comp(1)| across the 64 result elements,")
print("      after the burst ends (last 10 snapshots).")
print("  A2. Bus envelope E(x) = rms_y |m.Comp(1)| and its log-linear decay fit +")
print("      r2 over stated x windows.")
print("A3 (threshold switching) cannot be obtained from this deck because the")
print("elements are coarse-grained to 100 x 50 nm; switching claims go to the")
print("published threshold values, not to this simulation.")


check                                                   result
six P0 phase bands                                      PASS
ncode := 6                                              PASS
64 NiFe result elements                                 PASS
elements detached (ext_scaleExchange S=0)               PASS
drive f0 = 1.70 GHz                                     PASS
CPW antenna loaded from data file                       PASS
coarse-graining caveat stated in header                 PASS
transverse component saved (m.Comp(1))                  PASS
P0/Ck separation stated                                 PASS

deck structural checks passed

Stage B status on this machine: the deck HAS BEEN RUN (MX130, 40 snapshots
of m.Comp(1)) via Downloads/gdsii_run/run_stageB.sh; the analysis is in
tools/analyze_stageB.py.

Metric definitions actually used (S):
  A1. Contrast of the time-averaged |m.Comp(1)| across the 64 result elements,
      after the burst ends (last 10 snapshots).
  A2. Bus envelope

In [10]:

# Read the Stage B analysis output if present.
sb = os.path.join(REPO, "data", "lab", "stageB_summary.json")
if os.path.exists(sb):
    summary = json.load(open(sb, encoding="utf-8"))
    print(json.dumps(summary, indent=2))
else:
    print("stageB_summary.json not present yet — run:")
    print("  wsl -d Ubuntu-24.04 -u root -- \\")
    print("    /root/eda/build-env/bin/python3 tools/analyze_stageB.py \\")
    print("    /root/mumax/idaten_stageB.out $(pwd)")


{
  "tag": "(S)",
  "snapshots": 40,
  "t_first_ns": 0.7160558754171367,
  "t_last_ns": 29.200183709294524,
  "bus_decay_fits": {
    "5-45": {
      "ell_um": 111.25424682408423,
      "r2": 0.0032134229835080186,
      "n": 800
    },
    "10-45": {
      "ell_um": 113.7295679488324,
      "r2": 0.0023581275451214356,
      "n": 700
    },
    "10-50": {
      "ell_um": 90.3734270042058,
      "r2": 0.005054992445400863,
      "n": 800
    },
    "15-45": {
      "ell_um": 99.26325495973143,
      "r2": 0.0023035196251410817,
      "n": 600
    },
    "5-90": {
      "ell_um": 1018.1759989180069,
      "r2": 0.00017906280577384415,
      "n": 1700
    },
    "10-95": {
      "ell_um": null,
      "r2": 2.1998590958194342e-05,
      "n": 1700
    }
  },
  "result_array_elements": 64,
  "result_array_contrast_over_elements": 0.9548933732488759,
  "n_elements_above_half_max": 9,
  "note": "Spatial pattern only. No switching threshold, reversal probability, or write-error rate is measure


### Device-element ledger (architecture paper §10 requirement)

Every proposed element states: what quantity changes, what launches it,
retention/reset, how the next stage detects it, energy + delay, and the
experiment that could disprove it.

| Element | Quantity that changes | Launched by | Retention / reset | Next-stage detection | Energy + delay | Disproof experiment |
|---|---|---|---|---|---|---|
| CPW launch (P₀ port) | microwave field h_rf | RF source | stateless | — | P_C1 = 12.6 µW (L); transit 3.26 ns over 40.96 µm (E) | no antenna coupling measured → P_C1 not transferable |
| YIG bus | spin-wave amplitude/phase | CPW burst | stateless | result array absorbs | 36 nW into precession (L); 3.26 ns transit (E) | no propagating mode at k₁ detected |
| P₀ phase grating | local wave phase | fixed geometry | fixed (not at runtime) | wavefield downstream | 0 runtime energy (fixed) (E) | measured H(k) ≠ designed grating phase |
| `C_k` config switches (MRAM bits) | wave-pipeline route via switch resistance/phase | input coding | non-volatile; reset by foundry SOT/eMRAM process (L) | wavefield | 0.05 pJ/bit (E) | measured switch-phase spread larger than the π-step budget |
| NiFe result element (`M_k` cell) | magnetisation orientation (up/down) | magnon-assisted reversal | non-volatile; reset by foundry SOT (or strain hypothesis, optional) | MRAM sense amplifier | 0.20 fJ/cell (E, drive-power term / 205 cells) | wave-absent threshold same as wave-present (K1 fails) |

**The `P₀` code and the `M_k` cell are the same physical device class**, so
the ledger is deliberately symmetric: the wave is the switch, the cell is
the state, and the code lives in the medium. Nothing here requires the
piezomagnetic strain-reset hypothesis, which is demoted to optional
(`docs/CREDIBILITY_PLAN.md` §3.3).



## 4. Stage C — the material portfolio, and the hematite question

**The portfolio on the identical geometry** (runner: Windows-side WSL, mumax³
3.12 / mumax⁺ 1.2.2):

| Material | Tool | α (ref) | Role |
|---|---|---|---|
| YIG | mumax³ | 9×10⁻⁴ patterned / 2.7×10⁻⁵ best (L) | the in-circuit baseline |
| Bi-YIG | mumax³ | ~1×10⁻⁴ (L) | near-zero damping comparison |
| Co₂₅Fe₇₅ | mumax³ | ~2×10⁻³ (L) | ferrimagnetic comparator |
| GdFeCoP | mumax³ | ~1×10⁻³ (L) | amorphous comparator |
| NiFe | mumax³ | ~1×10⁻² (L) | the saturation-magnetisation comparator |
| **hematite α-Fe₂O₃** | **mumax⁺ 1.2.2** | **1.1×10⁻⁵ (L)** | **the low-loss candidate; antiferromagnetic Néel vector, needs the AF model** |

**Why hematite matters (L):** Hamdi, Posva & Grundler, *Phys. Rev. Materials*
**7**, 054407 (2023): α = 1.1×10⁻⁵, v_g = 10–23 km/s, ℓ = 1.1 cm at 30 mT,
T_N ≈ 955 K, T_M ≈ 262 K, easy-plane basal anisotropy ≈ 1 Oe above T_M. No
net magnetisation, so the operating point is not bias-field set; but the
Néel-state readout needs XMLD-PEEM or a companion ferromagnet (a real cost,
stated in `docs/CREDIBILITY_PLAN.md` §3.3).

**The write-amplification trade (E):** a 1.1 cm hematite guide at pitch
200 nm holds ~55 000 cells — ~268× the precedent's 205 — at a transit time
of ~700 ns vs 3.26 ns. Hematite buys parallel width and pays in latency.
Which side wins for a given workload is a **workload question**, which is
why the workload is frozen first (architecture Gate 0) and why notebook 02
answers it on a concrete task.

**Status: Stage C mumax³ shoot-out RUN COMPLETE (19 Nov 2026, Kaggle T4,
kernel `danielvking/idaten-stage-c-mumax` v4; assets
`danielvking/idaten-stagec-assets`).** Identical 100 µm × 8 µm bare bus +
64 result elements, 1.70 GHz / 1 mT burst, 4 ns, per material
(YIG/Bi-YIG/CoFe/GdFeCoP/NiFe). Measured (S), in
`data/stageC_kaggle/stageC_summary.json` and
`figures/stageC_material_shootout.png`:

| Material | f_dom (S) | probe-RMS decay ℓ over 10–80 µm (S) | v_g (S) |
|---|---|---|---|
| YIG | 1.93 GHz (drive 1.70 GHz; Stage A measured 1.89 GHz) | ~63 µm fit, but far probe retains ~20 % of near — no strong attenuation within the bus | xcorr median ≈ 22 km/s, lag spread comparable to median → **not resolved** at this SNR/window |
| Bi-YIG | 1.93 GHz | ~42 µm fit; flat profile, weakest decay | not resolved |
| Co₂₅Fe₇₅ | 2.94 GHz | ~119 µm fit; mid-bus dip then rise | not resolved |
| GdFeCoP | 3.55 GHz | ~266 µm fit; flat | not resolved |
| NiFe | 7.68 GHz | ~60 µm fit; lowest amplitudes overall | not resolved |

Honest reading: f_dom tracks Msat (higher-Msat materials ring higher), and
no material attenuates the wave within the 100 µm bus at this drive — but
the 4 ns window at ~2 GHz gives only ~8 cycles at probe amplitudes
~10⁻⁴–10⁻³, so group velocity is **not resolved** from this run; a longer
run is the stated follow-up. **The hematite arm has since run as Stage D
on mumax⁺ 1.2.2 — see §4b below.**



## 4b. Stage D — the hematite arm, actually run (mumax⁺ 1.2.2 wheel, Kaggle T4, (S))

**Why a separate solver:** hematite is an antiferromagnet above its Morin
temperature, so the wave variable is the Néel vector **n**, and mumax³
(ferromagnetic LLG) cannot represent it. mumax⁺ 1.2.2 (PyPI wheel,
GPL-3.0-only) has the `Antiferromagnet` class. The wheel's embedded PTX
targets sm_52 only: the local MX130 (sm_61, CUDA 13 driver) cannot JIT it,
the Kaggle T4 (sm_75, CUDA 13.0) can — so all mumax⁺ runs go to Kaggle
(kernel `danielvking/idaten-hematite-mumaxplus` v6, deck dataset
`danielvking/idaten-hematite-assets`; deck
`tools/decks/idaten_hematite_mumaxplus.py`).

**Parameter mapping (derived from the v1.2.2 source `afmexchange.cu`,
not guessed):** `afmex_cell = −H_E·Ms·latcon²/4`, `afmex_nn = −A_T·Ms`.
(L) inputs: Ms = 9.0×10⁵ A/m, α = 1.1×10⁻⁵ (Hamdi, Posva & Grundler, PRM
**7**, 054407 (2023)), H_E = 1040 T (arXiv:2610.08030), latcon = 0.5 nm,
μ₀A = 9.153×10⁻⁶ T·µm² (arXiv:2212.11887), easy-plane H_a = 1 Oe, 30 mT
bias. Deviations: DMI = 0, demag off ((E) net M ~ 10⁻⁵ Ms).

**Geometry:** 100 µm × 8 µm × 100 nm bus, 100 nm cells, fixed dt = 0.2 ps,
8 ns, 200 probes at 0.5 µm spacing; z-stripe RF launch at x ≈ 1 µm, burst
0.5–3 ns. Two arms: **1.70 GHz** (below the (E) qFM gap 12.78 GHz) and
**15 GHz at 0.1 mT** (above the gap, linear regime).

**What came out (S), full numbers in `data/hematite_kaggle/hematite_summary.json`
and PROVENANCE §2.8:**

| Quantity | 1.70 GHz arm | 15 GHz arm | Reference |
|---|---|---|---|
| launcher-probe f_dom | — | **14.23 GHz** (drive 15 GHz) | drive (L) |
| global ring-down f_dom | 2.51 GHz (forced, below gap) | **12.97 GHz** | (E) gap 12.78 GHz → **1.5 % match** |
| end-field decay ℓ | no decay fit | **≈ 2477 µm** ≫ 100 µm bus | (L) 1.1 cm at 30 mT |
| v_g (xcorr, 198 probe pairs) | 149 ± 3339 m/s — not resolved | 221 ± 4113 m/s — not resolved | (E) 24.3 km/s, (L) 10–23 km/s |
| intracell angle min / post-burst mean | 2.33° @ 0.14 ns / 123° | 2.33° @ 0.14 ns / 123° | 180° = perfect AFM |

Reading: the **gap frequency is reproduced to 1.5 %**, which validates the
afmex_cell ↔ H_E mapping — the single most important credibility check for
this arm, because it says the simulated AFM exchange stiffness is the
literature one. Below the gap there is only a forced response (no
propagating branch), exactly as the dispersion predicts. Above the gap the
ring-down is a **standing/global mode** (arrival times simultaneous at all
200 probes), so v_g is **not resolved** from this window — the honest
follow-up is a longer run with a windowed launch. The intracell-angle
transient (2.33° at burst onset) means the drive is locally nonlinear;
linear-regime readings carry that caveat.


In [11]:

hd = os.path.join(REPO, "data", "hematite_kaggle", "hematite_summary.json")
if os.path.exists(hd):
    s = json.load(open(hd, encoding="utf-8"))
    print("solver:", s["solver"])
    print("v_g (E) max =", s["v_g_pred_E_m_s"], "m/s ; (L)", s["v_g_lit_L_m_s"])
    for name, r in s["runs"].items():
        if r["table_rows"] < 100:
            continue
        print("---", name)
        print("  drive %.2f GHz | gap (E) %.3f GHz | f_dom global %.4f GHz | f_dom probe0 %.4f GHz"
              % (r["drive_f0_GHz_L"], r["qfm_gap_E_GHz"],
                 r.get("f_dom_global_nz_GHz") or float("nan"),
                 r.get("f_dom_probe0_nz_GHz") or float("nan")))
        print("  l_env_field_um:", r.get("l_env_field_um"),
              "| v_g xcorr:", r.get("v_g_xcorr_m_s"), "+/-", r.get("v_g_xcorr_spread_m_s"),
              "m/s | angle min/max deg:", r.get("intracell_angle_min_deg"), r.get("intracell_angle_max_deg"))
else:
    print("hematite_summary.json not present yet — see PROVENANCE §2.8 for the Kaggle run")


solver: mumaxplus 1.2.2 PyPI wheel on Kaggle T4 (sm_75, CUDA 13 JIT of sm_52 PTX)
v_g (E) max = 24296.2 m/s ; (L) 10000-23000 (Hamdi PRM 7, 054407 2023)
--- hematite_full
  drive 1.70 GHz | gap (E) 12.782 GHz | f_dom global 2.5105 GHz | f_dom probe0 20.2929 GHz
  l_env_field_um: None | v_g xcorr: 149.4 +/- 3338.6 m/s | angle min/max deg: 2.327 174.539
--- hematite_full15
  drive 15.00 GHz | gap (E) 12.782 GHz | f_dom global 12.9707 GHz | f_dom probe0 14.2259 GHz
  l_env_field_um: 2477.31 | v_g xcorr: 221.3 +/- 4112.8 m/s | angle min/max deg: 2.327 169.395



## 5. Summary — what this notebook establishes

| Statement | Tag | Status |
|---|---|---|
| The authors' phase-diagram arrays load and reproduce the published MAMR-dip structure (see the local-minima table) | (L)/(S) | done here |
| The published 2023 result has not itself been re-run on this machine | — | **pending** (MX130) |
| The local material series has no reportable decay length | (S) | **done here — r² 0.001–0.004** |
| The CPW antenna is a diamond/lens-shaped flared launch at the x = 0 edge, not a y = 0 stripe | (S) | **corrected, `tools/probe_antenna.py`** |
| Stage B deck exists, was executed on the MX130, and its honest outcome (r² ≈ 0.002; element contrast 0.955) is recorded; no switching claim | (S)/(A) | run and reported; not yet run at full element resolution |
| Stage C shoot-out (5 materials, Kaggle T4): f_dom per material, no in-bus attenuation, v_g not resolved at this SNR | (S) | **done — §4, PROVENANCE §2.7** |
| Stage D hematite arm (mumax⁺ 1.2.2, Kaggle T4): gap matched 12.97 vs 12.78 GHz (E), ℓ > 2.5 mm in-bus, v_g not resolved (standing mode), nonlinear angle transient caveat | (S) | **done — §4b, PROVENANCE §2.8** |
| The write-amplification metric W = cells/E_event is the headline quantity | (E)/(L) | defined in `docs/CREDIBILITY_PLAN.md` §2 |
| Any claim of a fabricated or measured device | — | **not claimed anywhere in this repository** |

### The disproof experiment

Stage A fails if a re-run of the authors' deck, on adequate hardware, does
not produce the k₁ = 1.70 GHz / λ = 7.4 µm mode or the 33.4 → 12.9 mT
threshold reduction — at which point this project is a careful archive of a
published result, not a new one. Stage B fails if a thresholded write into
the result elements shows no contrast after the drive burst ends. Stage C
fails on the hematite arm if coherent magnons cannot be driven in a
patterned thin film at α within a factor of a few of the single-crystal
1.1×10⁻⁵ — which is the gate K6 in `docs/KILL_CRITERIA.md`.
